# Task 3：多家银行并排比较

对应 [3-多家银行并排比较.md](./3-多家银行并排比较.md)。

对所有通过质量门禁的银行股执行相同口径月度定投（5,000 元/月），横向对比收益、风险、分红稳定性与估值水平。全部数据从 Task 1 标准表读取。

## 1. 环境与参数

回测口径与 Task 2 一致：每月 5,000 元，1 日买入，100 股整数倍，分红再投资，不计手续费和分红税。固定样本图选用 5 只代表性银行。

In [ ]:
from __future__ import annotations

import importlib
import math
import os
import re
import socket
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


WORKING_DIR = Path.cwd().resolve()
LABS_DIR = None
for _candidate in [WORKING_DIR, *WORKING_DIR.parents]:
    if _candidate.name == "labs" and (_candidate / "pyproject.toml").is_file():
        LABS_DIR = _candidate
        break
    if (_candidate / "labs" / "pyproject.toml").is_file():
        LABS_DIR = _candidate / "labs"
        break

LAB_DIR = LABS_DIR / "01_银行股定投回测"


plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False


AS_OF_DATE = "2026-07-31"
HORIZONS = (10, 5, 3)
MONTHLY_AMOUNT = 5000
BUY_DAY = 1
LOT_SIZE = 100
DIVIDEND_REINVEST = True

FIXED_SAMPLE = [
    ("601398", "工商银行"),
    ("601939", "建设银行"),
    ("600036", "招商银行"),
    ("601288", "农业银行"),
    ("601166", "兴业银行"),
]

DATA_DIR = LAB_DIR / "data" / "lab2"
PRICE_DIR = DATA_DIR / "prices"
DIVIDEND_DIR = DATA_DIR / "dividends"
CHART_DIR = LAB_DIR / "data" / "charts" / "task3_compare"
RESULT_DIR = LAB_DIR / "data" / "results"
for path in (CHART_DIR, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print({"AS_OF_DATE": AS_OF_DATE, "HORIZONS": HORIZONS, "MONTHLY_AMOUNT": MONTHLY_AMOUNT})

## 2. 函数定义

以下函数从原 `bank_core.py` 内联到本 Notebook，实现回测核心逻辑。与 Task 2 共用同一套计算函数。

### 回测计算函数

本组函数实现月度定投回测的核心逻辑：构建标的总收益净值、计算 XIRR、执行按月买入和分红再投资。

- `normalize_symbol`：从任意输入提取 6 位证券代码，前补零。
- `build_total_return_history`：用不复权收盘价 + 每股现金分红构建标的总收益净值（用于回撤计算）。
- `xirr`：计算 XIRR（内部收益率），牛顿迭代法，容忍不规则现金流日期。
- `contribution_dates`：计算定投买入日（每月 1 日，非交易日顺延至下一个交易日）。
- `_shares_on_or_before`：查找登记日对应的持股快照（用于分红再投资时的持股数）。
- `_max_loss_duration_days`：账户资产低于累计投入的最长连续天数。
- `_strategy_max_drawdown`：现金流调整后策略净值（含分红再投资）的最大回撤。
- `BacktestOutput`：数据类，封装回测结果（summary/transactions/account_history/total_return_history）。
- `simulate_bank_dca`：单只银行月度定投回测主函数：按月买入、分红再投资、计算 9 组指标。

In [ ]:
def normalize_symbol(value: Any) -> str:
    digits = "".join(c for c in str(value) if c.isdigit())
    return digits[-6:].zfill(6)


def build_total_return_history(
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
) -> pd.DataFrame:
    """用不复权收盘价和每股现金分红构建标的总收益净值。"""
    frame = prices[["date", "close"]].copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.dropna().drop_duplicates("date", keep="last").sort_values("date")
    events = dividends[["ex_date", "cash_dividend_per_share"]].copy()
    events["ex_date"] = pd.to_datetime(events["ex_date"])
    events = events.groupby("ex_date", as_index=False)["cash_dividend_per_share"].sum()
    frame = frame.merge(events, left_on="date", right_on="ex_date", how="left")
    frame["cash_dividend_per_share"] = frame["cash_dividend_per_share"].fillna(0.0)
    frame["daily_total_return"] = (
        (frame["close"] + frame["cash_dividend_per_share"])
        / frame["close"].shift(1)
        - 1.0
    )
    frame.loc[frame.index[0], "daily_total_return"] = 0.0
    frame["total_return_nav"] = (1.0 + frame["daily_total_return"]).cumprod()
    frame["drawdown"] = (
        frame["total_return_nav"] / frame["total_return_nav"].cummax() - 1.0
    )
    return frame[
        [
            "date", "close", "cash_dividend_per_share", "daily_total_return",
            "total_return_nav", "drawdown",
        ]
    ]


def xirr(cashflows: Iterable[float], dates: Iterable[Any]) -> float:
    values = np.asarray(list(cashflows), dtype=float)
    timestamps = [pd.Timestamp(date) for date in dates]
    if len(values) != len(timestamps) or len(values) < 2:
        return np.nan
    if np.all(values >= 0) or np.all(values <= 0):
        return np.nan
    years = np.array(
        [(date - timestamps[0]).total_seconds() / (365.25 * 86400)
         for date in timestamps],
        dtype=float,
    )

    def npv(rate: float) -> float:
        return float(np.sum(values / np.power(1.0 + rate, years)))

    lower = -0.9999
    upper = 1.0
    lower_value = npv(lower)
    upper_value = npv(upper)
    while lower_value * upper_value > 0 and upper < 1_000_000:
        upper = upper * 2.0 + 1.0
        upper_value = npv(upper)
    if lower_value * upper_value > 0:
        return np.nan
    for _ in range(250):
        midpoint = (lower + upper) / 2.0
        midpoint_value = npv(midpoint)
        if abs(midpoint_value) < 1e-8:
            return float(midpoint)
        if lower_value * midpoint_value <= 0:
            upper = midpoint
        else:
            lower = midpoint
            lower_value = midpoint_value
    return float((lower + upper) / 2.0)


def contribution_dates(
    trading_dates: Iterable[Any],
    *,
    start_date: Any,
    end_date: Any,
    buy_day: int = 1,
) -> list[pd.Timestamp]:
    dates = pd.DatetimeIndex(pd.to_datetime(list(trading_dates))).sort_values().unique()
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    dates = dates[(dates >= start) & (dates <= end)]
    if len(dates) == 0:
        return []

    selected: list[pd.Timestamp] = [pd.Timestamp(dates[0])]
    first_period = pd.Timestamp(dates[0]).to_period("M")
    last_period = pd.Timestamp(dates[-1]).to_period("M")
    for period in pd.period_range(first_period + 1, last_period, freq="M"):
        target = period.start_time + pd.Timedelta(days=buy_day - 1)
        candidates = dates[
            (dates.to_period("M") == period) & (dates >= target)
        ]
        if len(candidates):
            selected.append(pd.Timestamp(candidates[0]))
    return selected


def _shares_on_or_before(
    snapshots: dict[pd.Timestamp, int],
    date: pd.Timestamp,
) -> int:
    eligible = [key for key in snapshots if key <= date]
    return snapshots[max(eligible)] if eligible else 0


def _max_loss_duration_days(account_history: pd.DataFrame) -> int:
    """账户资产低于累计投入的最长连续天数。"""
    if account_history.empty:
        return 0
    asset = account_history["account_asset"].astype(float).reset_index(drop=True)
    contribution = account_history["cumulative_contribution"].astype(float).reset_index(drop=True)
    underwater = (asset < contribution).fillna(False)
    if not underwater.any():
        return 0
    max_run = 0
    current_run = 0
    for flag in underwater:
        if flag:
            current_run += 1
            if current_run > max_run:
                max_run = current_run
        else:
            current_run = 0
    return int(max_run)


def _strategy_max_drawdown(account_history: pd.DataFrame) -> float:
    """现金流调整后策略净值回撤：剔除新增本金的影响。"""
    if account_history.empty:
        return np.nan
    data = account_history.copy()
    data["net_value"] = (
        data["account_asset"] - data["cumulative_contribution"].shift(1).fillna(0)
    )
    # 起点净值为 0，无法直接计算回撤，使用累计净流入作为基线
    data["strategy_nav"] = data["net_value"].cummax().where(
        data["net_value"] > 0, other=data["net_value"]
    )
    data["strategy_drawdown"] = (
        data["strategy_nav"] / data["strategy_nav"].cummax() - 1.0
    )
    return float(data["strategy_drawdown"].min())


@dataclass
class BacktestOutput:
    summary: dict[str, Any]
    transactions: pd.DataFrame
    account_history: pd.DataFrame
    total_return_history: pd.DataFrame


def simulate_bank_dca(
    *,
    symbol: str,
    name: str,
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
    listing_date: Any,
    as_of_date: str,
    horizon_years: int,
    monthly_amount: float = 5000.0,
    buy_day: int = 1,
    lot_size: int = 100,
    dividend_reinvest: bool = True,
) -> BacktestOutput:
    """单只银行月度定投回测。

    纯计算函数：输入不复权行情 + 已实施分红 DataFrame，输出交易流水、
    账户历史、标的总收益净值与汇总指标。不调用任何外部接口。

    回测口径（与本 Lab .md 一致）：
    - 每月投入固定金额，不足 1 手的零钱留现金；
    - 除权日按收盘价立即用分红再投资原标的；
    - 不计算手续费与分红税；
    - 上市晚于窗口起点或数据不完整时，比较指标返回 np.nan。
    """
    symbol = normalize_symbol(symbol)
    as_of = pd.Timestamp(as_of_date)
    requested_start = as_of - pd.DateOffset(years=horizon_years)
    listing = pd.Timestamp(listing_date)

    price = prices.copy()
    price["date"] = pd.to_datetime(price["date"])
    price = price[
        price["date"].le(as_of) & price["date"].ge(min(requested_start, listing))
    ].drop_duplicates("date", keep="last").sort_values("date")
    if price.empty:
        raise ValueError(f"{symbol} 在 {horizon_years} 年窗口内无行情")

    actual_start_target = max(requested_start, listing)
    available = price[price["date"].ge(actual_start_target)]
    if available.empty:
        raise ValueError(f"{symbol} 上市后至截止日无行情")
    start = pd.Timestamp(available["date"].iloc[0])
    end = pd.Timestamp(price["date"].max())
    price = price[price["date"].between(start, end)].reset_index(drop=True)
    full_horizon = bool(
        listing <= requested_start
        and start <= requested_start + pd.Timedelta(days=15)
    )

    schedule = set(
        contribution_dates(
            price["date"],
            start_date=start,
            end_date=end,
            buy_day=buy_day,
        )
    )
    dividend_events = dividends.copy()
    if dividend_events.empty:
        dividend_events = pd.DataFrame(
            columns=["record_date", "ex_date", "cash_dividend_per_share"]
        )
    dividend_events["record_date"] = pd.to_datetime(
        dividend_events["record_date"], errors="coerce"
    )
    dividend_events["ex_date"] = pd.to_datetime(
        dividend_events["ex_date"], errors="coerce"
    )
    dividend_events = dividend_events[
        dividend_events["ex_date"].between(start, end)
    ].copy()
    grouped_dividends = {
        date: group for date, group in dividend_events.groupby("ex_date")
    }

    cash = 0.0
    shares = 0
    cumulative_contribution = 0.0
    total_dividend = 0.0
    total_purchase_cost = 0.0
    snapshots: dict[pd.Timestamp, int] = {}
    transactions: list[dict[str, Any]] = []
    account_rows: list[dict[str, Any]] = []
    external_cashflows: list[float] = []
    external_dates: list[pd.Timestamp] = []

    def execute_buy(date: pd.Timestamp, close: float, trade_type: str) -> None:
        nonlocal cash, shares, total_purchase_cost
        buy_shares = int(cash // (close * lot_size)) * lot_size
        buy_amount = buy_shares * close
        cash -= buy_amount
        shares += buy_shares
        total_purchase_cost += buy_amount
        transactions.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "trade_type": trade_type,
                "buy_price": close,
                "buy_shares": buy_shares,
                "buy_amount": buy_amount,
                "fee": 0.0,
                "remaining_cash": cash,
                "cumulative_shares": shares,
                "cumulative_contribution": cumulative_contribution,
                "repo_interest_accrued": 0.0,
                "dividend_received": 0.0,
                "dividend_tax": 0.0,
            }
        )

    for row in price.itertuples(index=False):
        date = pd.Timestamp(row.date)
        close = float(row.close)
        dividend_received_today = 0.0

        if date in grouped_dividends:
            for event in grouped_dividends[date].itertuples(index=False):
                record_date = (
                    pd.Timestamp(event.record_date)
                    if pd.notna(event.record_date)
                    else date - pd.Timedelta(days=1)
                )
                eligible_shares = _shares_on_or_before(snapshots, record_date)
                dividend_cash = (
                    eligible_shares * float(event.cash_dividend_per_share)
                )
                cash += dividend_cash
                total_dividend += dividend_cash
                dividend_received_today += dividend_cash
            if dividend_reinvest and dividend_received_today > 0:
                execute_buy(date, close, "dividend_reinvest")
                transactions[-1]["dividend_received"] = dividend_received_today

        if date in schedule:
            cash += monthly_amount
            cumulative_contribution += monthly_amount
            external_cashflows.append(-monthly_amount)
            external_dates.append(date)
            execute_buy(date, close, "monthly_contribution")

        snapshots[date] = shares
        market_value = shares * close
        asset = market_value + cash
        account_profit_rate = (
            asset / cumulative_contribution - 1.0
            if cumulative_contribution > 0 else np.nan
        )
        account_rows.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "close": close,
                "shares": shares,
                "cash": cash,
                "market_value": market_value,
                "account_asset": asset,
                "cumulative_contribution": cumulative_contribution,
                "account_profit_rate": account_profit_rate,
                "dividend_received": dividend_received_today,
            }
        )

    account = pd.DataFrame(account_rows)
    transaction_frame = pd.DataFrame(transactions)
    ending_market_value = float(account["market_value"].iloc[-1])
    ending_asset = float(account["account_asset"].iloc[-1])
    ending_cash = float(account["cash"].iloc[-1])
    ending_shares = int(account["shares"].iloc[-1])
    external_cashflows.append(ending_asset)
    external_dates.append(end)
    annualized_return = xirr(external_cashflows, external_dates)

    total_history = build_total_return_history(price, dividend_events)
    volatility = float(
        total_history["daily_total_return"].iloc[1:].std(ddof=1) * math.sqrt(252)
    )
    average_buy_price = (
        total_purchase_cost / ending_shares if ending_shares else np.nan
    )
    current_profit_rate = (
        float(price["close"].iloc[-1]) / average_buy_price - 1.0
        if average_buy_price and not np.isnan(average_buy_price) else np.nan
    )
    total_return = (
        ending_asset / cumulative_contribution - 1.0
        if cumulative_contribution else np.nan
    )
    max_loss = float(account["account_profit_rate"].min())
    max_drawdown = float(total_history["drawdown"].min())
    max_loss_duration = _max_loss_duration_days(account)
    strategy_max_dd = _strategy_max_drawdown(account)

    summary = {
        "symbol": symbol,
        "name": name,
        "horizon": f"{horizon_years}Y",
        "requested_start_date": requested_start,
        "start_date": start,
        "end_date": end,
        "listing_date": listing,
        "full_horizon": full_horizon,
        "contribution_months": len(schedule),
        "total_contribution": cumulative_contribution,
        "ending_shares": ending_shares,
        "ending_cash": ending_cash,
        "ending_market_value": ending_market_value,
        "ending_asset": ending_asset,
        "total_dividend": total_dividend,
        "total_return": total_return if full_horizon else np.nan,
        "xirr": annualized_return if full_horizon else np.nan,
        "average_buy_price": average_buy_price if full_horizon else np.nan,
        "current_profit_rate": current_profit_rate if full_horizon else np.nan,
        "max_drawdown": max_drawdown if full_horizon else np.nan,
        "max_loss_vs_contribution": max_loss if full_horizon else np.nan,
        "strategy_max_drawdown": strategy_max_dd if full_horizon else np.nan,
        "max_loss_duration_days": max_loss_duration if full_horizon else np.nan,
        "volatility": volatility if full_horizon else np.nan,
        "dividend_reinvest": dividend_reinvest,
    }
    return BacktestOutput(
        summary=summary,
        transactions=transaction_frame,
        account_history=account,
        total_return_history=total_history.assign(
            symbol=symbol, name=name, horizon=f"{horizon_years}Y"
        ),
    )

## 3. 读取 Task 1 标准表与质量门禁

In [ ]:
security_info = pd.read_csv(DATA_DIR / "security_info.csv", parse_dates=["list_date", "manual_review_as_of"])
quality_report_path = DATA_DIR / "quality_report.csv"
quality_report = pd.read_csv(quality_report_path) if quality_report_path.is_file() else pd.DataFrame()

bank_universe = security_info[security_info["manual_verified"] | security_info["f10_is_bank"]].copy()
print(f"证券基本信息：{len(security_info)} 行")
print(f"通过人工/F10 核验的银行：{len(bank_universe)}")

if not quality_report.empty and "BLOCKED_DATA_QUALITY" in quality_report.columns:
    blocked_symbols = set(quality_report.loc[quality_report["BLOCKED_DATA_QUALITY"], "symbol"])
    bank_universe = bank_universe[~bank_universe["symbol"].isin(blocked_symbols)]
    print(f"质量门禁剔除：{len(blocked_symbols)} 只，剩余 {len(bank_universe)} 只")
else:
    print("质量报告未生成或无 BLOCKED_DATA_QUALITY 字段，跳过剔除")

display(bank_universe[["symbol", "name", "exchange", "list_date"]].head())

## 4. 运行全部银行 × 3 档周期回测

In [ ]:
all_summaries = []
all_account_histories = {}
fixed_sample_set = {s for s, _ in FIXED_SAMPLE}

for row in bank_universe.itertuples(index=False):
    symbol = row.symbol
    name = row.name
    price_path = PRICE_DIR / f"{symbol}_daily_raw.parquet"
    dividend_path = DIVIDEND_DIR / f"{symbol}_dividend.parquet"
    if not price_path.is_file():
        print(f"跳过 {symbol} {name}：不复权行情缺失")
        continue
    prices = pd.read_parquet(price_path)
    prices["date"] = pd.to_datetime(prices["date"])
    dividends = pd.read_parquet(dividend_path) if dividend_path.is_file() else pd.DataFrame(
        columns=["ex_date", "record_date", "cash_dividend_per_share"]
    )
    if not dividends.empty:
        dividends["ex_date"] = pd.to_datetime(dividends["ex_date"])
        dividends["record_date"] = pd.to_datetime(dividends["record_date"])

    is_fixed = symbol in fixed_sample_set
    for horizon in HORIZONS:
        try:
            output = simulate_bank_dca(
                symbol=symbol, name=name, prices=prices, dividends=dividends,
                listing_date=row.list_date, as_of_date=AS_OF_DATE,
                horizon_years=horizon, monthly_amount=MONTHLY_AMOUNT,
                buy_day=BUY_DAY, lot_size=LOT_SIZE, dividend_reinvest=DIVIDEND_REINVEST,
            )
            all_summaries.append(output.summary)
            if is_fixed and horizon == 10:
                all_account_histories[symbol] = output.account_history
        except Exception as error:
            print(f"跳过 {symbol} {horizon}Y：{type(error).__name__}: {error}")

summary_df = pd.DataFrame(all_summaries)
print(f"\n回测完成：{len(summary_df)} 行（{len(bank_universe)} 只银行 × {len(HORIZONS)} 档周期）")
display(summary_df.head())

## 5. 核心指标横向对比表

5 项核心指标：XIRR、最大回撤（策略净值 + 相对本金）、最长亏损时间、分红稳定性、估值水平。

In [ ]:
core_columns = [
    "symbol", "name", "horizon", "full_horizon",
    "xirr", "strategy_max_drawdown", "max_loss_vs_contribution",
    "max_loss_duration_days",
]
core_metrics = summary_df[core_columns].copy()
core_metrics["valuation_pe_ttm"] = pd.NA
core_metrics["valuation_pb"] = pd.NA
display(core_metrics.sort_values(["horizon", "xirr"], ascending=[True, False]))

## 6. 辅助指标表

7 项辅助指标：累计投入、最终金额、最终持股、累计分红、逆回购收益、期末闲置现金、资金利用率。

In [ ]:
aux_columns = [
    "symbol", "name", "horizon",
    "total_contribution", "ending_asset", "ending_shares",
    "total_dividend", "ending_cash",
]
aux_metrics = summary_df[aux_columns].copy()
aux_metrics["repo_interest"] = 0.0
aux_metrics["utilization"] = (
    (aux_metrics["total_contribution"] - aux_metrics["ending_cash"])
    / aux_metrics["total_contribution"]
)
display(aux_metrics.head(10))

## 7. 分红稳定性子指标

连续派息年数、每股分红年化波动率、近 5 年分红 CAGR。

In [ ]:
def compute_dividend_stability(dividends: pd.DataFrame, as_of: str) -> dict:
    if dividends.empty:
        return {"consecutive_years": 0, "annual_volatility": pd.NA, "cagr_5y": pd.NA}
    as_of_ts = pd.Timestamp(as_of)
    events = dividends.copy()
    events["year"] = pd.to_datetime(events["ex_date"]).dt.year
    annual = events.groupby("year")["cash_dividend_per_share"].sum().sort_index()
    years = annual.index.tolist()
    consecutive = 0
    for year in range(as_of_ts.year, min(years) - 1, -1) if years else []:
        if year in annual.index:
            consecutive += 1
        else:
            break
    vol = float(annual.std()) if len(annual) >= 2 else pd.NA
    recent = annual.tail(5)
    cagr = float((recent.iloc[-1] / recent.iloc[0]) ** (1 / (len(recent) - 1)) - 1) if len(recent) >= 2 and recent.iloc[0] > 0 else pd.NA
    return {"consecutive_years": consecutive, "annual_volatility": vol, "cagr_5y": cagr}

dividend_stability_rows = []
for row in bank_universe.itertuples(index=False):
    symbol = row.symbol
    dividend_path = DIVIDEND_DIR / f"{symbol}_dividend.parquet"
    if not dividend_path.is_file():
        continue
    dividends = pd.read_parquet(dividend_path)
    stability = compute_dividend_stability(dividends, AS_OF_DATE)
    stability["symbol"] = symbol
    stability["name"] = row.name
    dividend_stability_rows.append(stability)

dividend_stability = pd.DataFrame(dividend_stability_rows)
display(dividend_stability.sort_values("consecutive_years", ascending=False).head(10))

## 8. 图表

6 张图：XIRR 排名、总回报净值路径、风险收益散点、滚动胜率热力图（口径与 Task 4 一致）、分红稳定性三联图、估值分位数对比。

In [ ]:
# 图1：固定样本 XIRR 排名柱状图（5 只 × 3 档周期）
fixed_symbols = [s for s, _ in FIXED_SAMPLE]
fixed_df = summary_df[summary_df["symbol"].isin(fixed_symbols) & summary_df["full_horizon"]].copy()

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(15, 5), sharey=True)
for i, horizon in enumerate(HORIZONS):
    ax = axes[i] if len(HORIZONS) > 1 else axes
    subset = fixed_df[fixed_df["horizon"] == f"{horizon}Y"].sort_values("xirr", ascending=False)
    bars = ax.bar(subset["name"], subset["xirr"], color="steelblue")
    ax.set_title(f"{horizon}Y XIRR")
    ax.set_ylabel("XIRR")
    ax.grid(True, alpha=0.3, axis="y")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("固定样本银行 XIRR 排名")
plt.tight_layout()
plt.savefig(CHART_DIR / "01_xirr_ranking.png", dpi=120)
plt.show()

In [ ]:
# 图2：总回报净值路径（5 只银行归一化净值，起点=100）
fig, ax = plt.subplots(figsize=(12, 6))
for symbol, name in FIXED_SAMPLE:
    if symbol not in all_account_histories:
        continue
    account = all_account_histories[symbol].copy()
    account["nav"] = account["account_asset"] / account["cumulative_contribution"] * 100
    ax.plot(account["date"], account["nav"], label=name, linewidth=1.5)

ax.axhline(100, color="gray", linestyle="--", alpha=0.5)
ax.set_title("固定样本银行 10Y 总回报净值路径（起点=100）")
ax.set_xlabel("日期")
ax.set_ylabel("归一化净值")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "02_total_return_nav.png", dpi=120)
plt.show()

In [ ]:
# 图3：风险收益散点图（10Y，气泡大小=max_loss_duration_days）
scatter_df = fixed_df[fixed_df["horizon"] == "10Y"].copy()

fig, ax = plt.subplots(figsize=(10, 7))
sizes = scatter_df["max_loss_duration_days"].fillna(0).clip(lower=1) * 2
scatter = ax.scatter(
    scatter_df["strategy_max_drawdown"], scatter_df["xirr"],
    s=sizes, c=range(len(scatter_df)), cmap="viridis", alpha=0.7, edgecolors="black"
)
for _, row in scatter_df.iterrows():
    ax.annotate(row["name"], (row["strategy_max_drawdown"], row["xirr"]),
                xytext=(5, 5), textcoords="offset points", fontsize=9)
ax.set_xlabel("策略净值最大回撤")
ax.set_ylabel("XIRR")
ax.set_title("固定样本银行 10Y 风险-收益（气泡=最长亏损天数）")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "03_risk_return_scatter.png", dpi=120)
plt.show()

In [ ]:
# 图4：滚动窗口胜率热力图（口径与 Task 4 一致，完整计算见 4-银行股滚动回测.ipynb）
print("图4 滚动窗口胜率热力图：口径与 Task 4 一致，完整计算见 4-银行股滚动回测.ipynb")
print("此处展示固定样本 10Y 窗口 XIRR 是否为正：")
fixed_10y = fixed_df[fixed_df["horizon"] == "10Y"][["name", "xirr", "full_horizon"]].copy()
fixed_10y["xirr_positive"] = fixed_10y["xirr"] > 0
display(fixed_10y)

In [ ]:
# 图5：分红稳定性三联图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fixed_stability = dividend_stability[dividend_stability["symbol"].isin(fixed_symbols)].copy()
fixed_stability = fixed_stability.set_index("symbol").reindex([s for s, _ in FIXED_SAMPLE]).reset_index()

axes[0].bar(fixed_stability["name"], fixed_stability["consecutive_years"], color="steelblue")
axes[0].set_title("连续派息年数")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(fixed_stability["name"], fixed_stability["annual_volatility"], color="orange")
axes[1].set_title("每股分红年化波动率")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(True, alpha=0.3, axis="y")

axes[2].bar(fixed_stability["name"], fixed_stability["cagr_5y"], color="green")
axes[2].set_title("近 5 年分红 CAGR")
axes[2].tick_params(axis="x", rotation=30)
axes[2].grid(True, alpha=0.3, axis="y")

plt.suptitle("固定样本银行分红稳定性")
plt.tight_layout()
plt.savefig(CHART_DIR / "05_dividend_stability.png", dpi=120)
plt.show()

In [ ]:
# 图6：估值分位数对比（待 Task 1 估值表补齐）
print("图6 估值分位数对比：待 Task 1 估值表补齐后生成")

## 9. 保存结果

In [ ]:
core_path = RESULT_DIR / "task3_core_metrics.csv"
aux_path = RESULT_DIR / "task3_auxiliary_metrics.csv"
stability_path = RESULT_DIR / "task3_dividend_stability.csv"

core_metrics.to_csv(core_path, index=False, encoding="utf-8-sig")
aux_metrics.to_csv(aux_path, index=False, encoding="utf-8-sig")
dividend_stability.to_csv(stability_path, index=False, encoding="utf-8-sig")

assert len(summary_df) > 0, "回测结果为空"
assert len(core_metrics) > 0, "核心指标表为空"
chart_files = list(CHART_DIR.glob("*.png"))
assert len(chart_files) >= 5, f"应生成至少 5 张图，实际 {len(chart_files)} 张"
print(f"Task 3 验收通过")
print(f"- {len(summary_df)} 行回测结果")
print(f"- {len(core_metrics)} 行核心指标")
print(f"- {len(chart_files)} 张图表生成")